In [1]:
import pandas as pd
import numpy as np
import pickle

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, roc_auc_score)

In [8]:
df = pd.read_csv(r"C:\Users\prana\OneDrive\Desktop\New folder\student_placement_prediction_dataset_2026.csv")

In [9]:
print("Dataset Shape:", df.shape)

Dataset Shape: (100000, 26)


In [10]:
print("\nDataset Columns:")
print(df.columns.tolist())


Dataset Columns:
['student_id', 'age', 'gender', 'cgpa', 'branch', 'college_tier', 'internships_count', 'projects_count', 'certifications_count', 'coding_skill_score', 'aptitude_score', 'communication_skill_score', 'logical_reasoning_score', 'hackathons_participated', 'github_repos', 'linkedin_connections', 'mock_interview_score', 'attendance_percentage', 'backlogs', 'extracurricular_score', 'leadership_score', 'volunteer_experience', 'sleep_hours', 'study_hours_per_day', 'placement_status', 'salary_package_lpa']


In [11]:
print("\nFirst 5 Rows:")
print(df.head())


First 5 Rows:
   student_id  age  gender  cgpa branch college_tier  internships_count  \
0           1   24    Male  7.53     IT       Tier 2                  4   
1           2   21    Male  7.92    CSE       Tier 2                  1   
2           3   22  Female  8.60    EEE       Tier 1                  0   
3           4   24    Male  6.68    CSE       Tier 1                  0   
4           5   20  Female  8.43     IT       Tier 3                  1   

   projects_count  certifications_count  coding_skill_score  ...  \
0               6                     1           99.238568  ...   
1               3                     6           80.966123  ...   
2               1                     1           49.177184  ...   
3               2                     2           79.359084  ...   
4               4                     3           65.018573  ...   

   mock_interview_score  attendance_percentage  backlogs  \
0             72.647009              77.463863         2   
1    

In [12]:
print("\nDataset Information:")
df.info()


Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 26 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   student_id                 100000 non-null  int64  
 1   age                        100000 non-null  int64  
 2   gender                     100000 non-null  object 
 3   cgpa                       100000 non-null  float64
 4   branch                     100000 non-null  object 
 5   college_tier               100000 non-null  object 
 6   internships_count          100000 non-null  int64  
 7   projects_count             100000 non-null  int64  
 8   certifications_count       100000 non-null  int64  
 9   coding_skill_score         100000 non-null  float64
 10  aptitude_score             100000 non-null  float64
 11  communication_skill_score  100000 non-null  float64
 12  logical_reasoning_score    100000 non-null  float64
 13  hackatho

In [13]:
print("\nMissing Values:")
print(df.isnull().sum())


Missing Values:
student_id                   0
age                          0
gender                       0
cgpa                         0
branch                       0
college_tier                 0
internships_count            0
projects_count               0
certifications_count         0
coding_skill_score           0
aptitude_score               0
communication_skill_score    0
logical_reasoning_score      0
hackathons_participated      0
github_repos                 0
linkedin_connections         0
mock_interview_score         0
attendance_percentage        0
backlogs                     0
extracurricular_score        0
leadership_score             0
volunteer_experience         0
sleep_hours                  0
study_hours_per_day          0
placement_status             0
salary_package_lpa           0
dtype: int64


In [14]:
print("\nPlacement Status Distribution:")
print(df["placement_status"].value_counts())


Placement Status Distribution:
placement_status
Placed        54459
Not Placed    45541
Name: count, dtype: int64


In [15]:
print("\nPlacement Status Percentage:")
print(
    df["placement_status"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)


Placement Status Percentage:
placement_status
Placed        54.46
Not Placed    45.54
Name: proportion, dtype: float64


4. DATA PREPARATION: 
student_id is only an identifier and should not be used
as a predictive feature.
salary_package_lpa must also be excluded because salary is
 known only after placement and would cause data leakage.

In [ ]:
df_model = df.drop(
    columns=["student_id", "salary_package_lpa"]).copy()

5. Target Encoding

In [21]:
df_model["placement_status"] = df_model["placement_status"].map({
    "Placed": 1,
    "Not Placed": 0
})

6. Separate Features & Target

In [22]:
X = df_model.drop(columns=["placement_status"])
y = df_model["placement_status"]

In [23]:
print("\nFeature Shape:", X.shape)
print("Target Shape:", y.shape)


Feature Shape: (100000, 23)
Target Shape: (100000,)


In [24]:
print("\nFeatures:")
for column in X.columns:
    print("-", column)


Features:
- age
- gender
- cgpa
- branch
- college_tier
- internships_count
- projects_count
- certifications_count
- coding_skill_score
- aptitude_score
- communication_skill_score
- logical_reasoning_score
- hackathons_participated
- github_repos
- linkedin_connections
- mock_interview_score
- attendance_percentage
- backlogs
- extracurricular_score
- leadership_score
- volunteer_experience
- sleep_hours
- study_hours_per_day


7. Identify Feature Types

In [25]:
categorical_features = [
    "gender",
    "branch",
    "college_tier",
    "volunteer_experience"
]

In [26]:
numerical_features = [
    column for column in X.columns
    if column not in categorical_features
]


In [27]:
print("\nCategorical Features:")
print(categorical_features)


Categorical Features:
['gender', 'branch', 'college_tier', 'volunteer_experience']


In [28]:
print("\nNumerical Features:")
print(numerical_features)


Numerical Features:
['age', 'cgpa', 'internships_count', 'projects_count', 'certifications_count', 'coding_skill_score', 'aptitude_score', 'communication_skill_score', 'logical_reasoning_score', 'hackathons_participated', 'github_repos', 'linkedin_connections', 'mock_interview_score', 'attendance_percentage', 'backlogs', 'extracurricular_score', 'leadership_score', 'sleep_hours', 'study_hours_per_day']


8. Train-Test Split


In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [30]:
print("\nTraining Samples:", X_train.shape[0])
print("Testing Samples:", X_test.shape[0])


Training Samples: 80000
Testing Samples: 20000


9. Preprocessing


In [31]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            StandardScaler(),
            numerical_features
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

10. Random Forest Model


In [32]:
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

11. Create Complete ML Pipeline


In [33]:
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ]
)

12. Train Model


In [34]:
print("\nTraining model...")

pipeline.fit(
    X_train,
    y_train
)

print("Model training completed successfully!")


Training model...
Model training completed successfully!


13. Predictions


In [35]:
y_pred = pipeline.predict(X_test)

y_probability = pipeline.predict_proba(X_test)[:, 1]

14. Model Evaluation


In [36]:
accuracy = accuracy_score(
    y_test,
    y_pred
)

roc_auc = roc_auc_score(
    y_test,
    y_probability
)

In [39]:
print("\n================================================")
print("MODEL PERFORMANCE")
print("================================================")

print(f"\nAccuracy: {accuracy:.4f}")
print(f"Accuracy (%): {accuracy * 100:.2f}%")

print(f"\nROC-AUC Score: {roc_auc:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "Not Placed",
            "Placed"
        ]
    )
)
print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        y_pred
    )
)


MODEL PERFORMANCE

Accuracy: 0.5557
Accuracy (%): 55.57%

ROC-AUC Score: 0.5772

Classification Report:
              precision    recall  f1-score   support

  Not Placed       0.51      0.50      0.51      9108
      Placed       0.59      0.60      0.60     10892

    accuracy                           0.56     20000
   macro avg       0.55      0.55      0.55     20000
weighted avg       0.55      0.56      0.56     20000


Confusion Matrix:
[[4567 4541]
 [4345 6547]]


15. Feature Importance


In [43]:
trained_model = pipeline.named_steps["model"]
trained_preprocessor = pipeline.named_steps["preprocessor"]

feature_names = trained_preprocessor.get_feature_names_out()

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": trained_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print("\n================================================")
print("TOP 20 FEATURE IMPORTANCE")
print("================================================")

print(
    feature_importance.head(20).to_string(index=False)
)


TOP 20 FEATURE IMPORTANCE
                             Feature  Importance
       numerical__coding_skill_score    0.076820
     numerical__mock_interview_score    0.072324
           numerical__aptitude_score    0.066577
  numerical__logical_reasoning_score    0.066411
numerical__communication_skill_score    0.064025
         numerical__leadership_score    0.062106
    numerical__extracurricular_score    0.060035
                     numerical__cgpa    0.056743
     numerical__linkedin_connections    0.056456
    numerical__attendance_percentage    0.055100
                 numerical__backlogs    0.048799
              numerical__sleep_hours    0.046427
      numerical__study_hours_per_day    0.046126
           numerical__projects_count    0.040044
        numerical__internships_count    0.035410
             numerical__github_repos    0.025860
                      numerical__age    0.021315
     numerical__certifications_count    0.020399
  numerical__hackathons_participated    0.

16. Save Complete Pipeline


In [44]:
with open(
    "student_placement_model.pkl",
    "wb"
) as file:

    pickle.dump(
        pipeline,
        file
    )

print("\nModel saved as:")
print("student_placement_model.pkl")


Model saved as:
student_placement_model.pkl


17. Save Feature Information


In [45]:
model_features = {
    "all_features": X.columns.tolist(),
    "categorical_features": categorical_features,
    "numerical_features": numerical_features
}

with open(
    "model_features.pkl",
    "wb"
) as file:

    pickle.dump(
        model_features,
        file
    )

print("\nFeature information saved as:")
print("model_features.pkl")


Feature information saved as:
model_features.pkl


18. Test Model With One Student



In [46]:
sample_student = X_test.iloc[[0]]

sample_prediction = pipeline.predict(
    sample_student
)[0]

sample_probability = pipeline.predict_proba(
    sample_student
)[0]

print("\n================================================")
print("SAMPLE STUDENT PREDICTION")
print("================================================")

print(
    "\nStudent Details:"
)

print(
    sample_student.to_string(index=False)
)

if sample_prediction == 1:

    print("\nPrediction: PLACED")

else:

    print("\nPrediction: NOT PLACED")

print(
    f"\nNot Placed Probability: "
    f"{sample_probability[0] * 100:.2f}%"
)

print(
    f"Placed Probability: "
    f"{sample_probability[1] * 100:.2f}%"
)



SAMPLE STUDENT PREDICTION

Student Details:
 age gender  cgpa branch college_tier  internships_count  projects_count  certifications_count  coding_skill_score  aptitude_score  communication_skill_score  logical_reasoning_score  hackathons_participated  github_repos  linkedin_connections  mock_interview_score  attendance_percentage  backlogs  extracurricular_score  leadership_score volunteer_experience  sleep_hours  study_hours_per_day
  20 Female  8.49     IT       Tier 2                  0               1                     1           66.062462        80.58324                  64.281335                53.816048                        0             7                   865             66.958081              88.520095         1              45.993977         88.192645                  Yes          6.8                  5.4

Prediction: NOT PLACED

Not Placed Probability: 57.57%
Placed Probability: 42.43%


19. Placement Probability Interpretation



In [47]:

placement_probability = sample_probability[1] * 100

print("\nPlacement Probability:")
print(f"{placement_probability:.2f}%")

if placement_probability >= 80:

    print("Risk Level: LOW")
    print("Student has a high predicted placement probability.")

elif placement_probability >= 60:

    print("Risk Level: MODERATE")
    print("Student has a moderate predicted placement probability.")

elif placement_probability >= 40:

    print("Risk Level: HIGH")
    print("Student may require additional placement support.")

else:

    print("Risk Level: VERY HIGH")
    print("Student should receive early intervention and career support.")


Placement Probability:
42.43%
Risk Level: HIGH
Student may require additional placement support.


20. Final Model Summary



In [48]:
print("\n================================================")
print("PROJECT MODEL BUILDING COMPLETED")
print("================================================")

print(f"Dataset Records: {len(df):,}")
print(f"Total Features Used: {len(X.columns)}")
print(f"Training Records: {len(X_train):,}")
print(f"Testing Records: {len(X_test):,}")
print(f"Model Accuracy: {accuracy * 100:.2f}%")
print(f"ROC-AUC: {roc_auc:.4f}")

print("\nGenerated Files:")
print("1. student_placement_model.pkl")
print("2. model_features.pkl")

print("\nThese files can be used later in the Streamlit application.")


PROJECT MODEL BUILDING COMPLETED
Dataset Records: 100,000
Total Features Used: 23
Training Records: 80,000
Testing Records: 20,000
Model Accuracy: 55.57%
ROC-AUC: 0.5772

Generated Files:
1. student_placement_model.pkl
2. model_features.pkl

These files can be used later in the Streamlit application.
